In [70]:
import pandas as pd
import numpy as np

from sklearn.metrics import mean_squared_error
import lightgbm as lgb
import plotly.express as px


In [ ]:

sales = pd.read_csv("wroclaw_retail.csv", parse_dates=["date"])
products = pd.read_csv("wroretail_products.csv")
stores = pd.read_csv("wroretail_stores.csv")
holidays = pd.read_csv("polish_holidays_2021_2024.csv", parse_dates=["date"])


sales = sales.sort_values(["store_id", "product_id", "date"]).reset_index(drop=True)


In [ ]:

df = sales.merge(products, on="product_id", how="left")


df = df.merge(stores, on="store_id", how="left")


df = df.merge(
    holidays,
    on="date",
    how="left"
)


df["is_public_holiday"] = df["is_public_holiday"].fillna(0)
df["is_shopping_boost"] = df["is_shopping_boost"].fillna(0)
df["category_boost"] = df["category_boost"].fillna(0)
df["holiday_name"] = df["holiday_name"].fillna("none")

df.head()


,date,store_id,product_id,sales,product_name,category,min_price,max_price,base_popularity,store_name,city,store_type,opening_year,sales_multiplier,holiday_name,is_public_holiday,is_shopping_boost,category_boost
0,2022-01-01,1,1,712.05,Telewizor Samsung 55,Elektronika,1200,2500,1.2,WroRetail Wroclaw Centrum,Wroclaw,hipermarket,2015,1.5,Nowy Rok,1.0,0.0,0
1,2022-01-02,1,1,798.06,Telewizor Samsung 55,Elektronika,1200,2500,1.2,WroRetail Wroclaw Centrum,Wroclaw,hipermarket,2015,1.5,none,0.0,0.0,0
2,2022-01-03,1,1,482.66,Telewizor Samsung 55,Elektronika,1200,2500,1.2,WroRetail Wroclaw Centrum,Wroclaw,hipermarket,2015,1.5,none,0.0,0.0,0
3,2022-01-04,1,1,483.14,Telewizor Samsung 55,Elektronika,1200,2500,1.2,WroRetail Wroclaw Centrum,Wroclaw,hipermarket,2015,1.5,none,0.0,0.0,0
4,2022-01-05,1,1,619.95,Telewizor Samsung 55,Elektronika,1200,2500,1.2,WroRetail Wroclaw Centrum,Wroclaw,hipermarket,2015,1.5,none,0.0,0.0,0


In [ ]:
df["day_of_week"] = df["date"].dt.weekday
df["month"] = df["date"].dt.month
df["quarter"] = df["date"].dt.quarter
df["year"] = df["date"].dt.year

df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)


In [74]:
def season_pl(month):
    if month in [12, 1, 2]:
        return "zima"
    if month in [3, 4, 5]:
        return "wiosna"
    if month in [6, 7, 8]:
        return "lato"
    return "jesien"

df["season"] = df["month"].apply(season_pl).astype("category")


In [75]:
dow = df.groupby("day_of_week")["sales"].mean().reset_index()

px.bar(
    dow,
    x="day_of_week",
    y="sales",
    title="Średnia sprzedaż vs dzień tygodnia"
).show()


In [76]:
month = df.groupby("month")["sales"].mean().reset_index()

px.line(
    month,
    x="month",
    y="sales",
    title="Średnia sprzedaż vs miesiąc"
).show()


In [77]:
for lag in [1, 7, 28]:
    df[f"sales_lag_{lag}"] = (
        df
        .groupby(["store_id", "product_id"])["sales"]
        .shift(lag)
    )


In [78]:
df["rolling_mean_7"] = (
    df.groupby(["store_id", "product_id"])["sales"]
    .shift(1)
    .rolling(7)
    .mean()
)

df["rolling_mean_28"] = (
    df.groupby(["store_id", "product_id"])["sales"]
    .shift(1)
    .rolling(28)
    .mean()
)

df["rolling_std_7"] = (
    df.groupby(["store_id", "product_id"])["sales"]
    .shift(1)
    .rolling(7)
    .std()
)


In [79]:
df["avg_sales_product"] = (
    df.groupby("product_id")["sales"]
    .shift(1)
    .expanding()
    .mean()
)


In [80]:
df["avg_sales_store"] = (
    df.groupby("store_id")["sales"]
    .shift(1)
    .expanding()
    .mean()
)


In [81]:
df["avg_sales_category_dow"] = (
    df.groupby(["category", "day_of_week"])["sales"]
    .shift(1)
    .expanding()
    .mean()
)


In [82]:
df["product_percentile_in_category"] = (
    df.groupby(["category", "date"])["sales"]
    .shift(1)
    .rank(pct=True)
)


In [83]:
df_model = df.dropna().copy()

target = "sales"

features = [
    "day_of_week", "month", "quarter", "year", "is_weekend",
    "is_public_holiday", "is_shopping_boost", "category_boost",
    "base_popularity", "sales_multiplier",
    "sales_lag_1", "sales_lag_7", "sales_lag_28",
    "rolling_mean_7", "rolling_mean_28", "rolling_std_7",
    "avg_sales_product", "avg_sales_store",
    "avg_sales_category_dow",
    "product_percentile_in_category"
]

df_model = pd.get_dummies(
    df_model,
    columns=["season", "store_type", "category"],
    drop_first=True
)

X = df_model[features + [c for c in df_model.columns if c.startswith(("season_", "store_type_", "category_"))]]
y = df_model[target]


In [84]:
split_idx = int(len(df_model) * 0.8)

X_train = X.iloc[:split_idx]
X_test = X.iloc[split_idx:]
y_train = y.iloc[:split_idx]
y_test = y.iloc[split_idx:]


In [85]:
baseline_features = ["day_of_week", "month", "is_weekend"]

Xb_train = X_train[baseline_features]
Xb_test = X_test[baseline_features]

baseline = lgb.LGBMRegressor(random_state=42)
baseline.fit(Xb_train, y_train)

baseline_pred = baseline.predict(Xb_test)
baseline_rmse = mean_squared_error(y_test, baseline_pred)
baseline_rmse


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000348 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 22
[LightGBM] [Info] Number of data points in the train set: 26956, number of used features: 3
[LightGBM] [Info] Start training from score 371.125031


46122.42597693682

In [ ]:

print(X_train.dtypes)

for c in X_train.columns:
    if pd.api.types.is_object_dtype(X_train[c]):
        X_train[c] = pd.to_numeric(X_train[c], errors="coerce").fillna(0)
        X_test[c] = pd.to_numeric(X_test[c], errors="coerce").fillna(0)




print(X_train.dtypes)
X_train = X_train.loc[:, ~X_train.columns.duplicated()]
X_test = X_test.loc[:, ~X_test.columns.duplicated()]
X_train["category_boost"] = pd.to_numeric(X_train["category_boost"], errors="coerce").fillna(0)
X_test["category_boost"] = pd.to_numeric(X_test["category_boost"], errors="coerce").fillna(0)




day_of_week                         int32
month                               int32
quarter                             int32
year                                int32
is_weekend                          int64
is_public_holiday                 float64
is_shopping_boost                 float64
category_boost                     object
base_popularity                   float64
sales_multiplier                  float64
sales_lag_1                       float64
sales_lag_7                       float64
sales_lag_28                      float64
rolling_mean_7                    float64
rolling_mean_28                   float64
rolling_std_7                     float64
avg_sales_product                 float64
avg_sales_store                   float64
avg_sales_category_dow            float64
product_percentile_in_category    float64
category_boost                     object
season_lato                          bool
season_wiosna                        bool
season_zima                       

In [87]:
model = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    random_state=42
)

model.fit(X_train, y_train)

pred = model.predict(X_test)
rmse = mean_squared_error(y_test, pred)
rmse


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.010100 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2608
[LightGBM] [Info] Number of data points in the train set: 26956, number of used features: 24
[LightGBM] [Info] Start training from score 371.125031


3003.7635063914554

In [89]:
importance = pd.DataFrame({
    "feature": model.feature_name_,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

import plotly.express as px

fig = px.bar(
    importance.head(20),
    x="importance",
    y="feature",
    orientation="h",
    title="Top 20 Feature Importance"
)
fig.show()
